In [0]:
-- DESCRIBE DETAILS OF TABLE (FORMAT, PARTITION COLUMNS, SIZE, LOCATION )
DESCRIBE DETAIL susep.bronze_ses_uf2;

In [0]:
-- SHOW PARTITIONS (COLUMNS AND VALUES)
SHOW PARTITIONS susep.bronze_ses_uf2;

In [0]:
-- GRUPOS DE RAMOS
SELECT
    gracodigo,
    granome
  FROM
    susep.bronze_ses_gruposramos
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_gruposramos
      )
    ORDER BY 1

In [0]:
 -- RAMOS
  SELECT
    coramo,
    noramo
  FROM
    susep.bronze_ses_ramos
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_ramos
      )

In [0]:
-- EXPLORATION RAMOS E GRUPOS
WITH cte_depara_ramos_grupo AS (
  SELECT DISTINCT
    ramos,
    gracodigo
  from
    susep.bronze_ses_uf2
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_uf2
      )
),
cte_ramos_grupo_agg as (
  SELECT
    ramos,
    gracodigo,
    count(*) over (partition by ramos) as total_grupo_ramos
  FROM
    cte_depara_ramos_grupo
),
cte_ramos AS (
  SELECT
    coramo,
    noramo
  FROM
    susep.bronze_ses_ramos
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_ramos
      )
),
cte_grupos_ramos AS (
  SELECT
    gracodigo,
    granome
  FROM
    susep.bronze_ses_gruposramos
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_gruposramos
      )
)
SELECT
  cte_ramos.*,
  trim(regexp_replace(cte_ramos.noramo, '^[0-9]+\\s*-\\s*', '')) as nome_ramo,
  cte_grupos_ramos.*,
   trim(regexp_replace(cte_grupos_ramos.granome, '^[0-9]+\\s*-\\s*', '')) as nome_grupo_ramo
FROM
  cte_ramos_grupo_agg
    left join cte_ramos
      on cte_ramos_grupo_agg.ramos = cte_ramos.coramo
    left join cte_grupos_ramos
      on cte_ramos_grupo_agg.gracodigo = cte_grupos_ramos.gracodigo
ORDER BY nome_ramo


In [0]:
-- RAMOS NOT EXISTS IN TABLE UF
SELECT distinct
    coramo 
FROM susep.bronze_ses_seguros
where coramo not in (
  SELECT DISTINCT
    ramos
  from
    susep.bronze_ses_uf2
)

In [0]:
-- RAMOS NOT EXISTS IN TABLE RAMOS
SELECT 
    coramo,
    min(damesano) as min_damesano,
    max(damesano) as max_damesano 
FROM susep.bronze_ses_seguros
where
  ingestion_batch_id
    = (
      select
        max(ingestion_batch_id)
      from
        susep.bronze_ses_seguros
    )
  and coramo not in (
    SELECT DISTINCT
      coramo
    from
      susep.bronze_ses_ramos
    where
      ingestion_batch_id
        = (
          select
            max(ingestion_batch_id)
          from
            susep.bronze_ses_ramos
        )
  )
  group by coramo

In [0]:
-- RAMOS DISTINTCS
SELECT distinct
    coramo 
FROM susep.bronze_ses_seguros
where
  ingestion_batch_id
    = (
      select
        max(ingestion_batch_id)
      from
        susep.bronze_ses_seguros
    )